# User data importer
This script is importing and formatting all data to be used by the platform

The source for the data is a file: Personregister.xlsx with the following tables
1. Personregister
2. Närvaro
3. Gradering baserat på ålder
4. Gradering baserat på närvaro

Slutresultatet en databas (json) med id, namn, ålder, närvaro, grad(dec), borde_grad(dec)

Vi läser öven in tabellen "Gradering i decimal till läsbart" och sparar i json som referens för webplatsen

In [5]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


## Import the tables

In [10]:
personer_df = pd.read_excel("G:\\Min enhet\\Styrelsens dokument\\Träning\\Gradering.xlsx","Personregister")          
narvaro_df = pd.read_excel("G:\\Min enhet\\Styrelsens dokument\\Träning\\Gradering.xlsx","Närvaro")
graderingsregler_df = pd.read_excel("G:\\Min enhet\\Styrelsens dokument\\Träning\\Gradering.xlsx","Graderingsregler")
grades_readable = pd.read_excel("G:\\Min enhet\\Styrelsens dokument\\Träning\\Gradering.xlsx","Grader läsbart")
grad_df = pd.read_excel("G:\\Min enhet\\Styrelsens dokument\\Träning\\Gradering.xlsx","Grad")

In [11]:
grad_df

,id,grade_decimal,name,grade_numeric,grade_text
0,201304174632,6.00,Adam Shehab,6,Vitt bälte
1,200402135917,6.00,Adrian Lewandowski,6,Vitt bälte
2,201709039372,6.00,Adrian Sahiti,6,Vitt bälte
3,201306064187,6.00,Agnes Näslund,6,Vitt bälte
4,201602287011,6.00,Albin Kupenberg,6,Vitt bälte
...,...,...,...,...,...
243,201411080763,6.00,Viva Cederek,6,Vitt bälte
244,201801229095,6.00,Walther Kriisa von Geijer,6,Vitt bälte
245,201908212895,6.00,Ward Alsamman,6,Vitt bälte
246,201601090291,4.75,Ylli Mavriqi,5-1,Gult bälte med ett orange streck


## Transform the imported tables

In [12]:
# personer_df = personer_df.drop(["Kön","c/o","Adress","Postnummer","Stad","Land","Mobiltelefon","Telefon hem","Telefon jobb","E-post","Målsman 1","Relation","E-post.1","Telefon","Målsman 2","Relation.1","E-post.2","Telefon.1","Skapad","Uppdaterad","Grupprekommendation","Övrigt","MedlemsNr", "Allergi"], axis=1)
personer_df = personer_df.drop(["Kön","c/o","Adress","Postnummer","Stad","Land","Mobiltelefon","Telefon hem","Telefon jobb","E-post","Målsman 1","Relation","E-post2","Telefon","Målsman 2","Relation3","E-post4","Telefon5","Skapad","Uppdaterad","Grupprekommendation","Övrigt","MedlemsNr", "Allergi"], axis=1)
personer_df = personer_df.rename(columns={
    "Personnummer":"id",
    "Kön":"sex",
    "Förnamn":"firstname",
    "Efternamn":"lastname",  
    "StartÅr":"started"    
  })
personer_df["id"] = (
    personer_df["id"]
    .astype(str)          # gör allt till str
    .str.replace(".0", "")  # tar bort float-svansar
    .str.strip()          # rensar whitespace
)


narvaro_df = narvaro_df.drop(["Namn","VT 2021","HT 2021","VT 2022","HT 2022","VT 2023","HT 2023","VT 2024","HT 2024","VT 2025","HT 2025" ], axis=1)
narvaro_df = narvaro_df.rename(columns={
    "ID":"id",
    "Namn":"name",
    "Aktiv":"active",
    "Totalt":"total"
  })
narvaro_df["active"] = narvaro_df["active"].str.strip().str.lower() == "ja"

narvaro_df["id"] = (
    narvaro_df["id"]
    .astype(str)          # gör allt till str
    .str.replace(".0", "")  # tar bort float-svansar
    .str.strip()          # rensar whitespace
)


grad_df = grad_df.drop(["name","grade_numeric", "grade_text"], axis=1)
grad_df = grad_df.rename(columns={"grade_decimal":"grade"  })

grad_df["id"] = (
    grad_df["id"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)




## Create aditional fields
1. Birthdate
2. Age

In [13]:
personer_df["birthdate"] = pd.to_datetime(personer_df["id"].str[:8], format="%Y%m%d", errors="coerce")
today = pd.Timestamp.today().normalize()
personer_df["age"] = (
    today.year - personer_df["birthdate"].dt.year
    - (
        (today.month < personer_df["birthdate"].dt.month) |
        ((today.month == personer_df["birthdate"].dt.month) & (today.day < personer_df["birthdate"].dt.day))
    )
)

In [14]:
personer_df

,id,firstname,lastname,started,birthdate,age
0,193705244139,Jan,Andersson,NaN,1937-05-24,88
1,193908034105,Lisbeth,Cederhag,NaN,1939-08-03,86
2,194003265230,Nils Olof,Lindberg,2025-04-03,1940-03-26,85
3,194204203501,Inger,Malmström,NaN,1942-04-20,83
4,194303013744,Gun,Pettersson,NaN,1943-03-01,82
...,...,...,...,...,...,...
286,202009269511,Liam,Bolejko,2024-09-08,2020-09-26,5
287,202012247777,Luca,Augustsson,2025-03-30,2020-12-24,4
288,202101290910,Benjamin,Jusufi,2025-02-16,2021-01-29,4
289,202103212490,Oliver,Mochtaghi - Svensson,2025-09-21,2021-03-21,4


## Sätt samman tabeller för personer och aktivitet

In [15]:
personer_df = personer_df.set_index("id")
narvaro_df   = narvaro_df.set_index("id")
grad_df   = grad_df.set_index("id")

merged_df = personer_df.merge(narvaro_df, on="id", how="inner")
merged_df = merged_df.join(grad_df, how="left")
merged_df = merged_df[merged_df["active"] != False]

In [16]:
merged_df


,firstname,lastname,started,birthdate,age,active,total,grade
id,,,,,,,,
193705244139,Jan,Andersson,NaN,1937-05-24,88,True,37,6.0
194003265230,Nils Olof,Lindberg,2025-04-03,1940-03-26,85,True,9,6.0
194303013744,Gun,Pettersson,NaN,1943-03-01,82,True,37,6.0
194511121230,Anders,Frick,NaN,1945-11-12,80,True,274,-3.0
194601104104,Margareta,Persson,2024-04-03,1946-01-10,79,True,27,6.0
...,...,...,...,...,...,...,...,...
202002259360,Linnea,Johansson Eek,2024-09-05,2020-02-25,5,True,10,6.0
202006243501,Ellen,Lange,2024-11-24,2020-06-24,5,True,15,6.0
202009263399,Leo,Bolejko,2024-09-08,2020-09-26,5,True,21,6.0


## Add should have grade
Based on age and activity (not skill)

In [17]:
belt_table = graderingsregler_df.sort_values("age")
def get_belt(age):
    row = belt_table[belt_table["age"] <= age].tail(1)
    if row.empty: return None   
    return row["grade"].iloc[0]
merged_df["should_grade_age"] = merged_df["age"].apply(get_belt)


active_table = graderingsregler_df.sort_values("activities")
def get_active_belt_value(total):
    row = active_table[active_table["activities"] <= total].tail(1)
    if row.empty:
        return None
    return row["grade"].iloc[0]
merged_df["should_grade_activity"] = merged_df["total"].apply(get_active_belt_value)

merged_df["should_have_grade"] = merged_df[["should_grade_age", "should_grade_activity"]].max(axis=1)
merged_df = merged_df.drop(["should_grade_activity","should_grade_age" ], axis=1)

In [18]:
merged_df

,firstname,lastname,started,birthdate,age,active,total,grade,should_have_grade
id,,,,,,,,,
193705244139,Jan,Andersson,NaN,1937-05-24,88,True,37,6.0,4.75
194003265230,Nils Olof,Lindberg,2025-04-03,1940-03-26,85,True,9,6.0,6.00
194303013744,Gun,Pettersson,NaN,1943-03-01,82,True,37,6.0,4.75
194511121230,Anders,Frick,NaN,1945-11-12,80,True,274,-3.0,1.75
194601104104,Margareta,Persson,2024-04-03,1946-01-10,79,True,27,6.0,5.00
...,...,...,...,...,...,...,...,...,...
202002259360,Linnea,Johansson Eek,2024-09-05,2020-02-25,5,True,10,6.0,6.00
202006243501,Ellen,Lange,2024-11-24,2020-06-24,5,True,15,6.0,6.00
202009263399,Leo,Bolejko,2024-09-08,2020-09-26,5,True,21,6.0,6.00


In [19]:
merged_df.to_csv("data.csv", index=True)

In [20]:
merged_df.to_json("Personregister.json", orient="records", force_ascii=False, indent=2)
